In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install mlflow dagshub

In [ ]:
import mlflow
import mlflow.sklearn
import dagshub
import pandas as pd
import numpy as np

In [ ]:
dagshub.init(repo_owner='slosa23', repo_name='ML-Assignment1', mlflow=True)

In [ ]:
model = mlflow.sklearn.load_model("models:/best-model/4")

In [ ]:
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

test_ids = test["Id"]

In [ ]:
test = test.drop("Id", axis=1)

num_cols = test.select_dtypes(include=["int64", "float64"]).columns
cat_cols = test.select_dtypes(include=["object"]).columns

test[num_cols] = test[num_cols].fillna(test[num_cols].median())
test[cat_cols] = test[cat_cols].fillna("None")

test["TotalSF"] = test["TotalBsmtSF"] + test["1stFlrSF"] + test["2ndFlrSF"]
test["Age"] = test["YrSold"] - test["YearBuilt"]

test["TotalBath"] = (
    test["FullBath"] +
    0.5 * test["HalfBath"] +
    test["BsmtFullBath"] +
    0.5 * test["BsmtHalfBath"]
)

test["TotalPorch"] = (
    test["OpenPorchSF"] +
    test["EnclosedPorch"] +
    test["3SsnPorch"] +
    test["ScreenPorch"]
)

test["HasGarage"] = (test["GarageArea"] > 0).astype(int)
test["HasBasement"] = (test["TotalBsmtSF"] > 0).astype(int)

for col in ["GrLivArea", "LotArea"]:
    test[col] = np.log1p(test[col])

test = pd.get_dummies(test)

In [ ]:
train_columns = model.feature_names_in_
test = test.reindex(columns=train_columns, fill_value=0)

In [ ]:
preds = model.predict(test)
preds = np.expm1(preds)

In [ ]:
submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": preds
})

submission.to_csv("submission.csv", index=False)

submission.head()